In [402]:
import cv2
import mediapipe as mp
import time
import numpy as np
from sklearn.neighbors import NearestNeighbors
import os

In [ ]:
MODEL_PATH = "face_landmarker.task"
IMAGE_FOLDER_PATH = "ref_images"
THRESH = 0.25
NTH_FRAME = 3        # run detector every N frames (or use time gating)
CANVAS_H = 720
CANVAS_W = 2000
PANEL_W  = CANVAS_W // 2
PAD_COLOR = (0, 0, 0)
frame_i = 0

In [404]:
BaseOptions = mp.tasks.BaseOptions
Vision = mp.tasks.vision
VisionRunningMode = Vision.RunningMode
FaceLandmarker = Vision.FaceLandmarker
FaceLandmarkerOptions = Vision.FaceLandmarkerOptions

In [405]:
data = np.load("ref_blendshapes.npz", allow_pickle=True)
labels = data["labels"]
vecs   = data["vectors"]            # already L2-normalized above
names  = data["blendshape_names"]   # for debugging/inspection

nn = NearestNeighbors(n_neighbors=1, metric='cosine').fit(vecs)

In [406]:
options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    output_face_blendshapes=True
)

In [407]:
cap = cv2.VideoCapture(1)  # choose your camera index
if not cap.isOpened():
    raise RuntimeError("Could not open webcam.")

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)

True

In [408]:
def draw_landmarks_on_frame(frame_bgr, result):
    """Draw dots for each landmark on each detected face."""
    colors = [(201, 141, 216), (14, 122, 30), (122, 14, 106)]

    if not result or not result.face_landmarks:
        return frame_bgr
    h, w = frame_bgr.shape[:2]
    for i, face in enumerate(result.face_landmarks[:2]):
        c = colors[i % len(colors)]
        for lm in face:
            x, y = int(lm.x * w), int(lm.y * h)
            cv2.circle(frame_bgr, (x, y), 2, c, -1)
    return frame_bgr

In [409]:
def resize_to_height(img, H):
    if img is None: return None
    h, w = img.shape[:2]
    if h == H: return img
    scale = H / float(h)
    return cv2.resize(img, (int(w*scale), H), interpolation=cv2.INTER_AREA)

In [ ]:
from Foundation import NSURL
from AVFoundation import AVAudioPlayer

# load once
url = NSURL.fileURLWithPath_("vine-boom.mp3")
player, err = AVAudioPlayer.alloc().initWithContentsOfURL_error_(url, None)
assert player is not None, err
player.prepareToPlay()  # prebuffer for instant start

def play_boom():
    player.stop()            # rewind if it’s mid-play
    player.setCurrentTime_(0)
    player.play()

In [411]:
ref_cache = {}
for lab in labels:
    p = os.path.join(IMAGE_FOLDER_PATH, lab)
    img = cv2.imread(p, cv2.IMREAD_COLOR)
    if img is not None:
        ref_cache[lab] = img
    else:
        print(f"[warn] could not read reference image: {p}")

In [412]:
def letterbox(img, tgt_h, tgt_w, color=(0,0,0)):
    """Return a tgt_h×tgt_w image with 'img' centered, preserving aspect ratio."""
    if img is None:  # empty placeholder
        return np.full((tgt_h, tgt_w, 3), color, dtype=np.uint8)
    h, w = img.shape[:2]
    scale = min(tgt_w / w, tgt_h / h)
    nw, nh = int(w*scale), int(h*scale)
    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    canvas = np.full((tgt_h, tgt_w, 3), color, dtype=np.uint8)
    y0 = (tgt_h - nh) // 2
    x0 = (tgt_w - nw) // 2
    canvas[y0:y0+nh, x0:x0+nw] = resized
    return canvas

In [413]:
# before loop (optional, to pin window size/position)
cv2.namedWindow("Meme", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Meme", CANVAS_W, CANVAS_H)
# cv2.moveWindow("Meme", 100, 100)  # position if you want

last_label = None
last_ref_panel = letterbox(None, CANVAS_H, PANEL_W, PAD_COLOR)  # start blank

t0 = time.monotonic()
with FaceLandmarker.create_from_options(options) as landmarker:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        ts = int((time.monotonic()-t0)*1000)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
        result = landmarker.detect_for_video(mp_image, ts)

        # draw landmarks on a copy of the live frame
        live = draw_landmarks_on_frame(frame.copy(), result)

        # ---- match logic (same as your code, sets match_label or None)
        match_label = None
        fb = getattr(result, "face_blendshapes", None)
        if fb:
            entry0 = fb[0]
            cats = entry0.categories if hasattr(entry0, "categories") else entry0
            b = np.array([c.score for c in cats], dtype=np.float32)
            b /= (np.linalg.norm(b)+1e-8)
            dist, idx = nn.kneighbors(b[None,:])
            d = float(dist[0,0])
            if d < THRESH:
                match_label = labels[int(idx[0,0])]
                cv2.putText(live, f"{match_label} ({d:.2f})", (20,40),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
            else:
                cv2.putText(live, f"No close match ({d:.2f})", (20,40),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0,200,200), 2)
        else:
            cv2.putText(live, "No blendshapes/face", (20,40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0,200,200), 2)

        # ---- build a constant-size canvas every frame
        left_panel  = letterbox(live, CANVAS_H, CANVAS_W, PAD_COLOR)

        # only rebuild right panel if the match changed
        if match_label and match_label != last_label and match_label in ref_cache:
            play_boom()
            ref_img = ref_cache[match_label]
            last_ref_panel = letterbox(ref_img, CANVAS_H, PANEL_W, PAD_COLOR)
            last_label = match_label
        elif match_label is None and last_label is None:
            # keep blank panel; optionally add text
            pass

        # optional labels on panels
        cv2.putText(left_panel,  "LIVE", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)
        if last_label:
            cv2.putText(last_ref_panel, f"REF: {last_label}", (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)
        else:
            cv2.putText(last_ref_panel, "REF: —", (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (200,200,200), 2)

        canvas = np.hstack([left_panel, last_ref_panel])
        cv2.imshow("Meme", canvas)

        if cv2.waitKey(1) & 0xFF in (27, ord('q')):
            break

cap.release()
cv2.destroyAllWindows()

I0000 00:00:1762299281.909335 23964340 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 83.1), renderer: Apple M1
W0000 00:00:1762299281.909616 23964340 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
W0000 00:00:1762299281.933437 24193655 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1762299281.956282 24193652 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [414]:
cap.release()
cv2.destroyAllWindows()

<!-- #80F5D4 -->